In [7]:
# ==========================================
# 1. CÀI ĐẶT & FIX LỖI "UNPICKLING"
# ==========================================
!pip uninstall -y transformers adapters huggingface_hub accelerate peft sentence-transformers
!pip install -q huggingface_hub==0.23.0 transformers==4.39.3 adapters==0.2.1 accelerate==0.27.2 datasets kaggle

import os, shutil, json, torch, time, nltk, numpy as np
from google.colab import drive
from transformers import AutoTokenizer
from datasets import Dataset
from adapters import AutoAdapterModel, AdapterConfig

os.environ["WANDB_DISABLED"] = "true"

# Cấp quyền cho các hàm Numpy
try:
    torch.serialization.add_safe_globals([np.core.multiarray._reconstruct, np.ndarray, np.dtype])
except AttributeError: pass

if not hasattr(torch, 'original_load_func'): torch.original_load_func = torch.load
def safe_load_override(*args, **kwargs):
    if 'weights_only' in kwargs: del kwargs['weights_only']
    return torch.original_load_func(*args, weights_only=False, **kwargs)
torch.load = safe_load_override

# ==========================================
# 2. THIẾT LẬP DỮ LIỆU & SPIDER-REALISTIC
# ==========================================
os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY'] = "0ba946628cb1f5acb76ecd357f590e95"
drive.mount('/content/drive')

FINAL_SAVE_PATH = "/content/drive/My Drive/SLM_Research/BART_Base_Spider_Houlsby"

print(">>> [1/4] Kiểm tra và tải dữ liệu...")
if not os.path.exists('spider_data'):
    !kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
    !unzip -q yale-universitys-spider-10-nlp-dataset.zip -d temp_spider
    shutil.move("temp_spider/spider", "spider_data")
    shutil.rmtree('temp_spider')
    !wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py
    !wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

if not os.path.exists('spider_data/spider-realistic.json'):
    !wget -q -O spider_data/spider-realistic.json "https://zenodo.org/records/5205322/files/spider-realistic.json?download=1"

# ==========================================
# 3. TIỀN XỬ LÝ SCHEMA VÀ LOAD DATASET
# ==========================================
with open('spider_data/tables.json', 'r') as f: table_data = json.load(f)
schema_map = {db['db_id']: db for db in table_data}

def get_crp_schema(db_id):
    if db_id not in schema_map: return ""
    db = schema_map[db_id]
    ddls = []
    for i, table_name in enumerate(db['table_names_original']):
        cols = [f"{c[1]} {db['column_types'][j]}" for j, c in enumerate(db['column_names_original']) if c[0] == i]
        ddls.append(f"CREATE TABLE {table_name} ({', '.join(cols)});")
    return " ".join(ddls)

def load_ds(path):
    with open(path, 'r', encoding='utf-8') as f: data = json.load(f)
    return Dataset.from_dict({"question": [x["question"] for x in data], "query": [x["query"] for x in data], "db_id": [x["db_id"] for x in data]})

print(">>> [2/4] Đang load tập Spider-Realistic...")
val_ds = load_ds("spider_data/spider-realistic.json")

# ==========================================
# 4. TỰ ĐỘNG KHÔI PHỤC VÀ LOAD MÔ HÌNH
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/bart-base"
print(f"\n>>> [3/4] Đang tải mô hình BART-Base và Adapter (Houlsby)...")

ADAPTER_LOAD_PATH = FINAL_SAVE_PATH
if os.path.exists(os.path.join(FINAL_SAVE_PATH, "spider_bart")):
    ADAPTER_LOAD_PATH = os.path.join(FINAL_SAVE_PATH, "spider_bart")

# --- ĐOẠN CODE "CHỮA LÀNH" KHÔI PHỤC CONFIG V2 ---
config_path = os.path.join(ADAPTER_LOAD_PATH, "adapter_config.json")
weights_path = os.path.join(ADAPTER_LOAD_PATH, "pytorch_adapter.bin")

def fix_adapter_config():
    print("⚠️ Phát hiện cấu hình Adapter bị sai lệch. Đang tự động tái tạo...")
    try:
        config = AdapterConfig.load("seq_bn")
        config_dict = config.to_dict() if hasattr(config, 'to_dict') else dict(config)

        # Bọc lại theo đúng cấu trúc thư viện yêu cầu
        proper_config = {
            "name": "spider_bart",
            "config": config_dict,
            "version": "0.2.1"
        }
        with open(config_path, "w") as f:
            json.dump(proper_config, f, indent=4)
        print("✅ Đã khôi phục file cấu hình đúng định dạng!")
    except Exception as e:
        print(f"❌ Lỗi khi khôi phục config: {e}")

if os.path.exists(weights_path):
    needs_fix = False
    if not os.path.exists(config_path):
        needs_fix = True
    else:
        # Kiểm tra xem file json đã có bọc "config" chưa
        with open(config_path, "r") as f:
            try:
                data = json.load(f)
                if "config" not in data:
                    needs_fix = True
            except:
                needs_fix = True

    if needs_fix:
        fix_adapter_config()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoAdapterModel.from_pretrained(MODEL_NAME)

# Load Adapter và Head
adapter_name = model.load_adapter(ADAPTER_LOAD_PATH, set_active=True)

model.to(device)
model.eval()
print("✅ Tải trọng số Adapter thành công!")

# ==========================================
# 5. INFERENCE & EVALUATION
# ==========================================
print("\n>>> [4/4] Bắt đầu chạy dự đoán trên tập Spider-Realistic (508 câu)...")
predictions, gold_lines = [], []

start_time = time.time()
total_items = len(val_ds)

for idx, item in enumerate(val_ds):
    input_text = f"translate to SQL: {item['question']} | Schema: {get_crp_schema(item['db_id'])}"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)

    predictions.append(tokenizer.decode(output[0], skip_special_tokens=True) + "\n")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

    if (idx + 1) % 50 == 0 or (idx + 1) == total_items:
        print(f"Đã dự đoán: {idx + 1}/{total_items} câu")

print(f"\n✅ Hoàn tất dự đoán trong {time.time() - start_time:.2f} giây.")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

print("\n>>> KẾT QUẢ ĐÁNH GIÁ TRÊN SPIDER-REALISTIC:")
!sed -i 's/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")/' evaluation.py
!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all

Found existing installation: transformers 4.39.3
Uninstalling transformers-4.39.3:
  Successfully uninstalled transformers-4.39.3
Found existing installation: adapters 0.2.1
Uninstalling adapters-0.2.1:
  Successfully uninstalled adapters-0.2.1
Found existing installation: huggingface-hub 0.23.0
Uninstalling huggingface-hub-0.23.0:
  Successfully uninstalled huggingface-hub-0.23.0
Found existing installation: accelerate 0.27.2
Uninstalling accelerate-0.27.2:
  Successfully uninstalled accelerate-0.27.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.23.0 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.23.0 which is incompatible.


/tmp/ipykernel_14869/3374315801.py:17: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  torch.serialization.add_safe_globals([np.core.multiarray._reconstruct, np.ndarray, np.dtype])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
>>> [1/4] Kiểm tra và tải dữ liệu...
>>> [2/4] Đang load tập Spider-Realistic...

>>> [3/4] Đang tải mô hình BART-Base và Adapter (Houlsby)...
⚠️ Phát hiện cấu hình Adapter bị sai lệch. Đang tự động tái tạo...
✅ Đã khôi phục file cấu hình đúng định dạng!
✅ Tải trọng số Adapter thành công!

>>> [4/4] Bắt đầu chạy dự đoán trên tập Spider-Realistic (508 câu)...
Đã dự đoán: 50/508 câu
Đã dự đoán: 100/508 câu
Đã dự đoán: 150/508 câu
Đã dự đoán: 200/508 câu
Đã dự đoán: 250/508 câu
Đã dự đoán: 300/508 câu
Đã dự đoán: 350/508 câu
Đã dự đoán: 400/508 câu
Đã dự đoán: 450/508 câu
Đã dự đoán: 500/508 câu
Đã dự đoán: 508/508 câu

✅ Hoàn tất dự đoán trong 329.93 giây.

>>> KẾT QUẢ ĐÁNH GIÁ TRÊN SPIDER-REALISTIC:
medium pred: SELECT Name,  Age FROM singer ORDER BY Age DESC LIMIT 1
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

eval_err_num:1
mediu